<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [يوري كاشنيتسكي](https://yorko.github.io). تمت الترجمة والتحرير بواسطة [كريستينا بوتسكو](https://www.linkedin.com/in/christinabutsko/)، و[نرسيس باجيان](https://www.linkedin.com/in/nersesbagiyan/)، و[يوليا كليموشينا](https://www.linkedin.com/in/yuliya-klimushina-7168a9139)، و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center> الموضوع 4. التصنيف الخطي والانحدار
## <center> الجزء الخامس. التحقق من الصحة ومنحنيات التعلم


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.linear_model import (LogisticRegression, LogisticRegressionCV,
                                  SGDClassifier)
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


الآن بعد أن أصبح لدينا فكرة عن التحقق من صحة النموذج، والتحقق المتبادل، والتنظيم. دعونا نفكر في السؤال الأكبر:
**ماذا تفعل إذا كانت جودة النموذج غير مرضية؟**
- هل يجب أن نجعل النموذج أكثر تعقيدا أم أبسط؟
- هل يجب أن نضيف المزيد من الميزات؟
- هل نحتاج ببساطة إلى المزيد من البيانات للتدريب؟
الإجابات على هذه الأسئلة ليست واضحة. على وجه الخصوص، في بعض الأحيان قد يؤدي النموذج الأكثر تعقيدًا إلى تدهور الأداء. وفي أحيان أخرى، لن تؤدي إضافة ملاحظات جديدة إلى إحداث تغييرات ملحوظة. في الواقع، فإن القدرة على اتخاذ القرار الصحيح واختيار الطريقة الصحيحة لتحسين النموذج هي ما يميز المحترف الجيد عن المحترف السيئ.



سنعمل على بياناتنا المتعلقة بتقلبات العملاء لدى مشغلي الاتصالات.


In [ ]:
data = pd.read_csv("../../data/telecom_churn.csv").drop("State", axis=1)
data["International plan"] = data["International plan"].map({"Yes": 1, "No": 0})
data["Voice mail plan"] = data["Voice mail plan"].map({"Yes": 1, "No": 0})

y = data["Churn"].astype("int").values
X = data.drop("Churn", axis=1).values


** سوف نقوم بتدريب الانحدار اللوجستي مع نزول التدرج العشوائي. وفي وقت لاحق من الدورة، سيكون لدينا مقال منفصل حول هذا الموضوع.**


In [ ]:
alphas = np.logspace(-2, 0, 20)
sgd_logit = SGDClassifier(loss="log", n_jobs=-1, random_state=17, max_iter=5)
logit_pipe = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2)),
        ("sgd_logit", sgd_logit),
    ]
)
val_train, val_test = validation_curve(
    logit_pipe, X, y, "sgd_logit__alpha", alphas, cv=5, scoring="roc_auc"
)


**كخطوة أولى، سنقوم بإنشاء منحنيات التحقق التي توضح مدى اختلاف الجودة (ROC-AUC) في مجموعات التدريب والاختبار مع معلمة التنظيم.**


In [ ]:
def plot_with_err(x, data, **kwargs):
    mu, std = data.mean(1), data.std(1)
    lines = plt.plot(x, mu, "-", **kwargs)
    plt.fill_between(
        x,
        mu - std,
        mu + std,
        edgecolor="none",
        facecolor=lines[0].get_color(),
        alpha=0.2,
    )


plot_with_err(alphas, val_train, label="training scores")
plot_with_err(alphas, val_test, label="validation scores")
plt.xlabel(r"$\alpha$")
plt.ylabel("ROC AUC")
plt.legend()
plt.grid(True);


الاتجاه واضح تمامًا وهو شائع جدًا.- بالنسبة للنماذج البسيطة، تكون أخطاء التدريب والتحقق متقاربة وكبيرة. يشير هذا إلى أن النموذج **غير مجهز**، مما يعني أنه لا يحتوي على عدد كافٍ من المعلمات.
- بالنسبة للنماذج المتطورة للغاية، تختلف أخطاء التدريب والتحقق بشكل كبير. يمكن تفسير ذلك من خلال **التركيب الزائد**. عندما يكون هناك عدد كبير جدًا من المعلمات أو عندما لا يكون التنظيم صارمًا بدرجة كافية، يمكن أن "تشتت انتباه" الخوارزمية بسبب الضجيج الموجود في البيانات وتفقد مسار الاتجاه العام.



### ما مقدار البيانات المطلوبة؟
كلما زاد عدد البيانات التي يستخدمها النموذج، كان ذلك أفضل. ولكن كيف نفهم ما إذا كانت البيانات الجديدة مفيدة في أي موقف معين؟ على سبيل المثال، هل من المنطقي إنفاق $N$ للمقيمين لمضاعفة مجموعة البيانات؟
نظرًا لأن البيانات الجديدة قد لا تكون متاحة، فمن المعقول تغيير حجم مجموعة التدريب ومعرفة كيف تعتمد جودة الحل على كمية بيانات التدريب. هذه هي الطريقة التي نحصل بها على **منحنيات التعلم**.
الفكرة بسيطة: نعرض الخطأ كدالة لعدد الأمثلة المستخدمة في التدريب. يتم تحديد معلمات النموذج مسبقًا.


In [ ]:
def plot_learning_curve(degree=2, alpha=0.01):
    train_sizes = np.linspace(0.05, 1, 20)
    logit_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("poly", PolynomialFeatures(degree=degree)),
            (
                "sgd_logit",
                SGDClassifier(n_jobs=-1, random_state=17, alpha=alpha, max_iter=5),
            ),
        ]
    )
    N_train, val_train, val_test = learning_curve(
        logit_pipe, X, y, train_sizes=train_sizes, cv=5, scoring="roc_auc"
    )
    plot_with_err(N_train, val_train, label="training scores")
    plot_with_err(N_train, val_test, label="validation scores")
    plt.xlabel("Training Set Size")
    plt.ylabel("AUC")
    plt.legend()
    plt.grid(True);


دعونا نرى ما نحصل عليه للنموذج الخطي. سنقوم بتعيين معامل التنظيم ليكون كبيرًا جدًا.


In [ ]:
plot_learning_curve(degree=2, alpha=10)

موقف نموذجي: بالنسبة لكميات صغيرة من البيانات، تكون الأخطاء بين مجموعات التدريب والتحقق المتبادل مختلفة تمامًا، مما يشير إلى التجاوز. بالنسبة لنفس النموذج ولكن مع كمية كبيرة من البيانات، "تتقارب" الأخطاء، مما يشير إلى عدم التجهيز.
 
إذا أضفنا المزيد من البيانات، فلن ينمو الخطأ في مجموعة التدريب. ومن ناحية أخرى، لن يتم تقليل الخطأ في بيانات الاختبار.
 
لذلك نرى أن الأخطاء "متقاربة" ولن يساعد إضافة بيانات جديدة. في الواقع هذه الحالة هي الأكثر إثارة للاهتمام بالنسبة للأعمال. من الممكن أن نزيد حجم مجموعة البيانات بمقدار 10 أضعاف، ولكن دون تغيير مدى تعقيد النموذج، قد لا تساعد هذه البيانات الإضافية. ولذلك فإن استراتيجية "اضبط مرة واحدة، ثم استخدم 10 مرات" قد لا تنجح.
 
ماذا يحدث إذا قمنا بتقليل معامل التنظيم إلى 0.05؟
 
نرى اتجاهًا جيدًا - تتقارب المنحنيات تدريجيًا، وإذا انتقلنا إلى اليمين، أي أضفنا المزيد من البيانات إلى النموذج، فيمكننا تحسين الجودة في مجموعة التحقق بشكل أكبر. 


In [ ]:
plot_learning_curve(degree=2, alpha=0.05)


الآن، ماذا لو جعلنا النموذج أكثر تعقيدًا عن طريق تحديد alpha = 1e-4؟
تمت ملاحظة التجهيز الزائد - تنخفض المساحة المخصصة للاستخدام (AUC) في كل من مجموعات التدريب والتحقق من الصحة.


In [ ]:
plot_learning_curve(degree=2, alpha=1e-4)


يمكن أن يساعد إنشاء هذه المنحنيات في فهم الطريق الذي يجب اتباعه وكيفية ضبط مدى تعقيد النموذج للبيانات الجديدة بشكل صحيح.



**الاستنتاجات حول منحنيات التعلم والتحقق:**- الخطأ في مجموعة التدريب لا يوضح شيئًا عن جودة النموذج في حد ذاته
- يظهر خطأ التحقق من الصحة مدى ملاءمة النموذج للبيانات (الاتجاه الحالي في البيانات) مع الاحتفاظ بالقدرة على التعميم على البيانات الجديدة
- **منحنى التحقق** هو رسم بياني يوضح نتائج التدريب ومجموعات التحقق اعتمادًا على **تعقيد النموذج**:
    + إذا كان المنحنيان قريبان من بعضهما البعض وكلا الخطأين كبيران فهذا علامة على *نقص التجهيز*
    + إذا كان المنحنيان بعيدان عن بعضهما البعض فهذا علامة على *التركيب الزائد*
- **منحنى التعلم** هو رسم بياني يوضح نتائج مجموعات التدريب والتحقق اعتمادًا على عدد الملاحظات:
    + إذا تقاربت المنحنيات، فإن إضافة بيانات جديدة لن يساعد، ومن الضروري تغيير مدى تعقيد النموذج 
    + إذا لم تتقارب المنحنيات، فإن إضافة بيانات جديدة يمكن أن يؤدي إلى تحسين النتيجة


### موارد مفيدة
- الطبق الرئيسي [الموقع](https://mlcourse.ai)، [مستودع الدورة](https://github.com/Yorko/mlcourse.ai)، ويوتيوب [القناة](https://www.youtube.com/watch?v=QKTuw4PNOsU&list=PLVlY_7IJCMJeRfZ68eVfEcu-UcN9BbwiX)
- متوسط ["قصة"](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-4-linear-classification-and-regression-44a41b9b5220) استنادًا إلى دفتر الملاحظات هذا
- مواد الدورة التدريبية باعتبارها [مجموعة بيانات Kaggle](https://www.kaggle.com/kashnitsky/mlcourse)
- إذا كنت تقرأ اللغة الروسية: [مقالة](https://habrahabr.ru/company/ods/blog/323890/) على حبراهابر مع ~ نفس المادة. و[محاضرة](https://youtu.be/oTXGQ-_oqvI) على اليوتيوب
- نظرة عامة لطيفة وموجزة على النماذج الخطية مقدمة في كتاب ["التعلم العميق"](http://www.deeplearningbook.org) (I. Goodfellow، Y. Bengio، و A. Courville).
- تتم تغطية النماذج الخطية عمليا في كل كتاب تعلم الآلة. نوصي بـ "التعرف على الأنماط والتعلم الآلي" (C. Bishop) و"التعلم الآلي: منظور احتمالي" (K. Murphy).
- إذا كنت تفضل نظرة شاملة على النموذج الخطي من وجهة نظر الإحصائي، فاطلع على "عناصر التعلم الإحصائي" (T. Hastie، R. Tibshirani، و J. Friedman).
- سيرشدك كتاب "التعلم الآلي أثناء العمل" (P. Harrington) عبر تطبيقات خوارزميات تعلم الآلة الكلاسيكية في لغة بايثون النقية.
- مكتبة [Scikit-learn](http://scikit-learn.org/stable/documentation.html). هؤلاء الرجال يعملون بجد لكتابة وثائق واضحة حقًا.
- Scipy 2017 [برنامج تعليمي لـ scikit-learn](https://github.com/amueller/scipy-2017-sklearn) بواسطة Alex Gramfort وAndreas Mueller.
- [دورة تعلم الآلة] (https://github.com/diefimov/MTH594_MachineLearning) إضافية بمواد جيدة جدًا.
- [تطبيقات](https://github.com/rushter/MLAlgorithms) للعديد من خوارزميات تعلم الآلة. البحث عن الانحدار الخطي والانحدار اللوجستي.